In [1]:
# this script downloads and processes the latest PDPASA and STPASA files

In [2]:
#1. libraries

import os
import getpass
import pandas as pd
import timeit
from io import BytesIO
import datetime
import re
from collections import namedtuple
import requests
import zipfile
from zipfile import ZipFile
import pickle
from pathlib import Path
import urllib.request
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup

print('OK')

OK


In [3]:
#2. preliminaries

def is_databricks():
    return "DATABRICKS_RUNTIME_VERSION" in os.environ


if is_databricks() == True:
    base = Path('/Volumes/exploration/bills_repository/files/')
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import to_utc_timestamp, to_timestamp
    df_duid_data = pd.read_parquet(base / "df_duid_LIVE.parquet") 
else:
    base = Path("C:/Users/BillNixey/OneDrive - Squadron Energy/Desktop/Working_files/New_model/test")
    df_duid_data = pd.read_csv(base / "df_duid_LIVE.csv") 
test_run = False 
if test_run == True:
    days=-10
else:
    days = 0


my_urls = ["https://www.nemweb.com.au/REPORTS/CURRENT/PDPASA/","https://www.nemweb.com.au/REPORTS/CURRENT/Short_Term_PASA_Reports/","https://www.nemweb.com.au/Reports/CURRENT/STPASA_DUIDAvailability/","https://www.nemweb.com.au/Reports/CURRENT/PDPASA_DUIDAvailability/"]
my_prefixes =  ['PUBLIC_PDPASA_','PUBLIC_STPASA_','PUBLIC_STPASA_DUIDAVAILABILITY_','PUBLIC_PDPASA_DUIDAVAILABILITY_']
regions = ['NSW1','QLD1','SA1','TAS1','VIC1']

pasa_columns = ["RUN_DATETIME","INTERVAL_DATETIME","REGIONID","DEMAND50","SS_SOLAR_UIGF","SS_WIND_UIGF"] #,"AGGREGATESCHEDULEDLOAD","AGGREGATECAPACITYAVAILABLE","UNCONSTRAINEDCAPACITY",'CONSTRAINEDCAPACITY','SURPLUSCAPACITY','UIGF','SEMISCHEDULEDCAPACITY'

print('done')



done


In [4]:
#3. functions

def nemfile_reader(nemfile_object):
    """
    Returns a dict containing a pandas dataframe each table in a nemfile.
    The fileobject needs to be unzipped csv (nemfile), and can be either a file or an
    an in stream fileobject.
    """
    table_dict = {}
    for line in nemfile_object.readlines():
        rows = line.decode().split(',')
        table = "{0}_{1}".format(rows[1], rows[2])

        #  new table
        if rows[0] == "I":
            table_dict[table] = line

        #  append data to each table
        elif rows[0] == "D":
            table_dict[table] += line

    return {table: pd.read_csv(BytesIO(table_dict[table]))
            for table in table_dict}


def latest_file(url,step_back,prefix,date_len): 
    my_list = []
    req = Request(url)
    a = urlopen(req).read()
    soup = BeautifulSoup(a, 'html.parser')
    x = (soup.find_all('a')) #read all on html page
    
    #this loop finds the most recent file
    for i in x:    
        file_name = i.extract().get_text()       #take only file names
        if(prefix in file_name)==True:            
                m=len(prefix)
                date = int(file_name[m:m+date_len]) #grabs the date and converts to an integer          
                my_list.append(date) #add item to array
        else:
            pass
    #print(my_list)
    if step_back != 0:
        now = datetime.datetime.now()
        one_week_ago = now - datetime.timedelta(days=step_back)
        s1 = str(one_week_ago)
        s2 = s1[:11]
        s2 = s2.replace(":", "")
        s2 = s2.replace("-", "")
        s2 = s2.replace(" ", "")
        my_value=int(s2+"0000")
        print(my_value)
    else:
        my_value = max(my_list) #find most recent report in array 
    location = my_list.index(my_value)    #position in array of latest date. Note 'zero' position at start of list.
    #the loop below downloads latest file a file and splits it into two df. One each for price and volume. Repeats for next most recent file.   
    my_file = x[location+len(x)-len(my_list)].extract().get_text() #This "len(x)-len(my_list)" takes into account 2 extra values in the x array
    return my_file



print('done')

done


In [5]:
#4. model

start = timeit.default_timer()

print('Starting...')

df2 = pd.DataFrame()
for n in range(2):
    url = my_urls[n]
    prefix = my_prefixes[n]
    file = latest_file(url,days,prefix,12)
    print(file)
    urllib.request.urlretrieve(url+file,base / file)
   
    with zipfile.ZipFile(base / file) as z:
        inner_name = z.namelist()[0]
        with z.open(inner_name) as f:
            result = nemfile_reader(f)
    
    df = pd.concat(result.values(), ignore_index=True)
    file_path = Path(base / file)
    file_path.unlink()   #remove file
    
    df = df[df.REGIONID.isin(regions)].reset_index(drop=True)
    df = df[pasa_columns]
    df2 = pd.concat([df2,df])

df2 = df2.sort_values(by=["RUN_DATETIME",'REGIONID'],ascending=False)
df2 = df2.drop_duplicates(subset=['INTERVAL_DATETIME','REGIONID'],keep='first')
df2 = df2.sort_values(by=['INTERVAL_DATETIME','REGIONID'],ascending=True).reset_index(drop=True)

if is_databricks() == True: 
    df_spark = spark.createDataFrame(df2)
    df_spark.write.mode("overwrite").saveAsTable("exploration.bills_repository.PASA")
else:
    df2.to_csv(base / 'PASA.csv')

end = timeit.default_timer()
print(end-start)

df2

Starting...
PUBLIC_PDPASA_202608121730_0000000532291763.zip
PUBLIC_STPASA_202608121700_0000000532292445.zip
19.771428700070828


,RUN_DATETIME,INTERVAL_DATETIME,REGIONID,DEMAND50,SS_SOLAR_UIGF,SS_WIND_UIGF
0,2026/08/12 17:30:00,2026/08/12 17:30:00,NSW1,9876.0,82.51,928.19
1,2026/08/12 17:30:00,2026/08/12 17:30:00,QLD1,7514.0,220.80,883.09
2,2026/08/12 17:30:00,2026/08/12 17:30:00,SA1,1647.0,73.28,1261.67
3,2026/08/12 17:30:00,2026/08/12 17:30:00,TAS1,1192.0,0.00,289.89
4,2026/08/12 17:30:00,2026/08/12 17:30:00,VIC1,7125.0,70.30,2602.57
...,...,...,...,...,...,...
1785,2026/08/12 17:00:00,2026/08/20 04:00:00,NSW1,7184.0,0.00,858.30
1786,2026/08/12 17:00:00,2026/08/20 04:00:00,QLD1,5679.0,0.00,560.36
1787,2026/08/12 17:00:00,2026/08/20 04:00:00,SA1,1364.0,0.00,407.56
1788,2026/08/12 17:00:00,2026/08/20 04:00:00,TAS1,1018.0,0.00,134.81


In [6]:
#5. model for PASA DUID

start = timeit.default_timer()

print('Starting...')
df2_parts = []
for n in [2,3]:
    url = my_urls[n]
    prefix = my_prefixes[n]
    
    file = latest_file(url, days, prefix, 12)
    print(file)
    
    urllib.request.urlretrieve(url + file, base / file)
    zip_path = base / file
    # CLOSES automatically after extraction
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(base)
    # delete zip
    zip_path.unlink()
    
    csv_file = Path(file).with_suffix(".CSV")
    
    df = pd.read_csv(base / csv_file, skiprows=1,skipfooter=1,engine='python')
    zip_path2 = base / csv_file
    zip_path2.unlink()
    
    df = df[['DUID','INTERVAL_DATETIME','GENERATION_MAX_AVAILABILITY','RUN_DATETIME']]# ,'LOAD_MAX_AVAILABILITY','GENERATION_PASA_AVAILABILITY','LOAD_PASA_AVAILABILITY'
    
    
    df = df.fillna(0)
    df2_parts.append(df)
df2 = pd.concat(df2_parts, ignore_index=True)


df2 = df2.sort_values(by=["RUN_DATETIME",'DUID'],ascending=False)
df2 = df2.drop_duplicates(subset=['INTERVAL_DATETIME','DUID'],keep='first')
df2 = df2.sort_values(by=['INTERVAL_DATETIME','DUID'],ascending=True).reset_index(drop=True)


df2 = pd.merge(df2,df_duid_data,how='left',on='DUID')

df2 = df2.dropna(subset=['REGIONID','FUEL'])

df2 = df2[df2.GENERATION_MAX_AVAILABILITY >0]

if is_databricks() == True: 

    df_spark = spark.createDataFrame(df2)
    df_spark.write.mode("overwrite").saveAsTable("exploration.bills_repository.PASA_DUID")
else:
    df2.to_csv(base / 'pasa_duid.csv',index=False)

end = timeit.default_timer()
print(end-start)
df2


Starting...
PUBLIC_STPASA_DUIDAVAILABILITY_202608121700_0000000532291108.zip
PUBLIC_PDPASA_DUIDAVAILABILITY_202608121730_0000000532291387.zip
5.814867200097069


,DUID,INTERVAL_DATETIME,GENERATION_MAX_AVAILABILITY,RUN_DATETIME,REGIONID,MLF,FUEL
0,ADPBA1,2026/08/12 17:30:00,6.000,2026/08/12 17:30:00,SA1,1.0033,Battery
1,ADPPV1,2026/08/12 17:30:00,1.852,2026/08/12 17:30:00,SA1,1.0033,Solar
2,AGLHAL,2026/08/12 17:30:00,142.000,2026/08/12 17:30:00,SA1,0.9628,Gas
3,AGLSOM,2026/08/12 17:30:00,126.000,2026/08/12 17:30:00,VIC1,0.9968,Gas
4,ALDGASF1,2026/08/12 17:30:00,2.547,2026/08/12 17:30:00,QLD1,0.8948,Solar
...,...,...,...,...,...,...,...
170403,YENDWF1,2026/08/20 04:00:00,44.296,2026/08/12 17:00:00,VIC1,0.9547,Wind
170404,YWPS1,2026/08/20 04:00:00,345.000,2026/08/12 17:00:00,VIC1,0.9658,Brown_Coal
170405,YWPS2,2026/08/20 04:00:00,320.000,2026/08/12 17:00:00,VIC1,0.9584,Brown_Coal
170406,YWPS3,2026/08/20 04:00:00,385.000,2026/08/12 17:00:00,VIC1,0.9584,Brown_Coal
